# Demo negFIN — Local
    
Clone repo, mở notebook bằng kernel Python 3.11+ đã cài `pandas` và `openpyxl`. Đặt Julia trong PATH hoặc cấu hình `NEGFIN_JULIA`. Chọn **Run All**.

Notebook chạy ví dụ toy, Online Retail (giỏ hàng) và Splice-junction (pattern vị trí đã cho theo lớp). Mọi frequent itemset do CLI negFIN của nhóm sinh ra. Dữ liệu UCI được kiểm SHA-256 trước khi dùng.

In [ ]:
VARIANT = "baseline"          # hoặc "optimized"
RETAIL_MINSUP = "0.01"        # chuỗi thập phân theo docs/baseline-contract.md
RETAIL_MINCONF = 0.5
SPLICE_MINSUP = "0.3"

## 1. Môi trường: mã nguồn, Julia, thư viện Python

In [ ]:
import hashlib, os, shutil, subprocess, sys, tomllib, urllib.request, zipfile
from pathlib import Path

def sh(*cmd, cwd=None, tail=6000):
    print("$", " ".join(map(str, cmd)))
    r = subprocess.run(list(map(str, cmd)), cwd=cwd, env={**os.environ, "PYTHONUTF8": "1"}, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout[-tail:], r.stderr[-tail:], sep="")
    r.check_returncode()
    return r.stdout

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download(url, destination, expected_sha256):
    """Download with a network timeout and verify bytes before publishing the file."""
    part = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "negFIN-course-demo/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=60) as response, part.open("wb") as output:
            shutil.copyfileobj(response, output, length=1 << 20)
        if sha256(part) != expected_sha256:
            raise ValueError(f"SHA-256 không khớp: {url}")
        part.replace(destination)
    finally:
        part.unlink(missing_ok=True)

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "run.jl").is_file() and (p / "data/applications/manifest.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook từ thư mục gốc repo hoặc một thư mục con.")
os.chdir(ROOT)
COMMIT = sh("git", "rev-parse", "HEAD").strip()

In [ ]:
JULIA = os.environ.get("NEGFIN_JULIA") or shutil.which("julia")
if JULIA is None:
    raise RuntimeError("Cài Julia >= 1.9 và đưa vào PATH hoặc đặt NEGFIN_JULIA.")
sh(JULIA, "--version")
try:
    import openpyxl
    import pandas as pd
except ImportError as exc:
    raise RuntimeError("Cài pandas và openpyxl trong môi trường kernel: python -m pip install pandas openpyxl") from exc
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| commit", COMMIT)

def negfin(inp, out, minsup):
    """Gọi CLI negFIN của nhóm; trả về stats TOML."""
    stats = Path(str(out).removesuffix(".txt") + ".stats.toml")
    sh(JULIA, "--startup-file=no", "--project=.", "run.jl", "--input", inp, "--output", out,
       "--minsup", minsup, "--variant", VARIANT, "--stats", stats)
    with open(stats, "rb") as f:
        return tomllib.load(f)

**Thư viện Python.** `openpyxl` giúp đọc workbook Excel của Online Retail. `pandas` đọc bảng, chuẩn bị dữ liệu Retail và hiển thị các luật kết hợp. Các bước khai thác itemset chạy bằng Julia qua `run.jl`.

## 2. Dữ liệu công khai

Nguồn, DOI, giấy phép (CC BY 4.0) và SHA-256 nằm trong `data/applications/manifest.toml`. File đã có và đúng checksum thì không tải lại.

In [ ]:
RAW = Path("data/applications/raw")
with open("data/applications/manifest.toml", "rb") as f:
    manifest = tomllib.load(f)["datasets"]

for d in manifest:
    target = RAW / d["path"]
    if not (target.exists() and sha256(target) == d["sha256"]):
        folder = RAW / d["name"]
        folder.mkdir(parents=True, exist_ok=True)
        z = folder / "download.zip"
        download(d["source_url"], z, d["zip_sha256"])
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
    assert sha256(target) == d["sha256"], f"SHA-256 {target} không khớp"
    print(f"OK {d['title']}: {target} ({d['license']}, doi:{d['doi']})")

## 3. Kiểm tra nhanh CLI trên ví dụ toy

In [ ]:
OUT = Path("results/demo")
OUT.mkdir(parents=True, exist_ok=True)
negfin("data/toy/example.txt", OUT / "toy.txt", "0.4")
def itemsets(path):
    return {line.strip() for line in open(path, encoding="utf-8") if line.strip()}
assert itemsets(OUT / "toy.txt") == itemsets("data/toy/example.expected.txt")
print("toy: khớp", len(itemsets(OUT / "toy.txt")), "itemset với kết quả mong đợi")

## 4. Ứng dụng 1 — Giỏ hàng (Online Retail)

Một hóa đơn là một transaction, item là `StockCode` sản phẩm. Bỏ hóa đơn hủy/điều chỉnh, `Quantity ≤ 0`, `UnitPrice ≤ 0` và mã không phải sản phẩm (xem `docs/applications-proposal.md`).
Luật `X ⇒ Y`: support = sup(X∪Y)/N, confidence = sup(X∪Y)/sup(X), lift = confidence/(sup(Y)/N).

In [ ]:
R = OUT / "retail"
sh(sys.executable, "apps/retail/prepare.py", "--input", RAW / "retail/Online Retail.xlsx", "--outdir", R)
retail_stats = negfin(R / "transactions.txt", R / f"itemsets_{RETAIL_MINSUP}.txt", RETAIL_MINSUP)
print({k: retail_stats[k] for k in ("n_transactions", "min_count", "n_patterns", "total_seconds")})

In [ ]:
sh(sys.executable, "apps/retail/rules.py", "--itemsets", R / f"itemsets_{RETAIL_MINSUP}.txt",
   "--stats", R / f"itemsets_{RETAIL_MINSUP}.stats.toml", "--mapping", R / "mapping.csv",
   "--output", R / "rules.csv", "--minconf", RETAIL_MINCONF)
rules = pd.read_csv(R / "rules.csv")
rules.head(10)[["antecedent_names", "consequent_names", "support_count", "support", "confidence", "lift"]]

## 5. Ứng dụng 2 — Sinh học (Splice-junction)

Một chuỗi DNA 60 base là một transaction; item là `p<vị trí>_<base>` (−30..−1, +1..+30). negFIN chạy riêng từng lớp: EI (donor), IE (acceptor), N (không phải điểm nối).
`growth` = support trong lớp / support trong các lớp còn lại.

In [ ]:
S = OUT / "splice"
sh(sys.executable, "apps/splice/prepare.py", "--input", RAW / "splice/splice.data", "--outdir", S)
splice_stats = {}
for c in ("EI", "IE", "N"):
    splice_stats[c] = negfin(S / f"transactions_{c}.txt", S / f"itemsets_{c}_{SPLICE_MINSUP}.txt", SPLICE_MINSUP)
    print(c, {k: splice_stats[c][k] for k in ("n_transactions", "min_count", "n_patterns")})

In [ ]:
for c in ("EI", "IE"):
    sh(sys.executable, "apps/splice/compare.py", "--target", c, "--itemsets", S / f"itemsets_{c}_{SPLICE_MINSUP}.txt",
       "--data", S, "--output", S / f"compare_{c}.csv", "--top", "8")

Kỳ vọng (consensus đã công bố): donor `…AG|GTRAGT…` ở EI và acceptor `…(Y)n CAG|G…` ở IE; lớp N không có pattern ở minsup 0,3. Diễn giải đầy đủ: `docs/applications-splice-results.md`, `docs/applications-retail-results.md`.

## 6. Thông tin tái lập

In [ ]:
print("commit  :", COMMIT)
print("julia   :", sh(JULIA, "--version").strip())
print("variant :", VARIANT)
print("retail  :", f"N={retail_stats['n_transactions']} minsup={RETAIL_MINSUP} itemsets={retail_stats['n_patterns']} minconf={RETAIL_MINCONF} rules={len(rules)}")
print("splice  :", f"minsup={SPLICE_MINSUP} " + " ".join(f"{c}:{s['n_transactions']}->{s['n_patterns']}" for c, s in splice_stats.items()))

if (RETAIL_MINSUP, RETAIL_MINCONF) == ("0.01", 0.5):
    assert (retail_stats["n_transactions"], retail_stats["n_patterns"], len(rules)) == (19773, 2026, 962)
if SPLICE_MINSUP == "0.3":
    assert {name: (s["n_transactions"], s["n_patterns"]) for name, s in splice_stats.items()} == {
        "EI": (767, 280), "IE": (768, 331), "N": (1655, 0)
    }